In [0]:
dbutils.library.restartPython()

In [0]:
# =========================
# INPUT WIDGETS
# =========================

dbutils.widgets.text(
    name="notebook_path",
    defaultValue="",
    label="Databricks Notebook Path"
)

dbutils.widgets.text(
    name="brd_file_name",
    defaultValue="BRD_Output",
    label="BRD File Name (without .docx)"
)

print("Widgets created")



Widgets created


In [0]:
# imports

from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat

from databricks_langchain import ChatDatabricks
from langchain_core.prompts import PromptTemplate

import base64
from datetime import datetime
import uuid


In [0]:
# =========================
# CONFIG
# =========================

VS_ENDPOINT_NAME = "brd-agent-vs-endpoint"

NOTEBOOK_INDEX = "main.doc_test.brd_notebook_index"
BRD_INDEX      = "main.doc_test.brd_gold_brd_index"

LLM_ENDPOINT = "databricks-gpt-5-2"

OUTPUT_VOLUME = "/Volumes/tmp/tmp/brd_agent/output"

TOP_K_NOTEBOOK = 25
TOP_K_BRD      = 10

print("Configuration loaded")


Configuration loaded


In [0]:
# loading system prompt from volume

SYSTEM_PROMPT_DIR = "/Volumes/tmp/tmp/brd_agent/system_prompt"

SYSTEM_PROMPT_FILES = [
    "01_identity.txt",
    "02_parsing_rules.txt",
    "03_brd_structure.txt",
    "04_docx_formatting.txt"
]

system_prompt_parts = []

for fname in SYSTEM_PROMPT_FILES:
    path = f"{SYSTEM_PROMPT_DIR}/{fname}"
    with open(path, "r") as f:
        content = f.read().strip()
        system_prompt_parts.append(content)

SYSTEM_PROMPT = "\n\n".join(system_prompt_parts)

print("System prompt loaded from 4 files")
print("Total characters:", len(SYSTEM_PROMPT))


System prompt loaded from 4 files
Total characters: 8700


In [0]:
# no silednt failures
assert "ANTI-HALLUCINATION" in SYSTEM_PROMPT, "System prompt integrity check failed"
assert "FINAL PROJECTION" in SYSTEM_PROMPT, "Mapping rules missing"

In [0]:
# read notebook
def read_notebook_content(notebook_path: str) -> str:
    w = WorkspaceClient()
    export = w.workspace.export(
        path=notebook_path,
        format=ExportFormat.SOURCE
    )
    return base64.b64decode(export.content).decode("utf-8")


In [0]:
# vector search retrieval
vsc = VectorSearchClient()


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [0]:
# notebook logic context
def retrieve_notebook_context(query: str):
    res = vsc.similarity_search(
        endpoint_name=VS_ENDPOINT_NAME,
        index_name=NOTEBOOK_INDEX,
        query_text=query,
        columns=["content"],
        num_results=TOP_K_NOTEBOOK
    )
    return "\n\n".join(r["content"] for r in res["result"]["data_array"])


In [0]:
# gold brd structure context

def retrieve_brd_context(query: str):
    res = vsc.similarity_search(
        endpoint_name=VS_ENDPOINT_NAME,
        index_name=BRD_INDEX,
        query_text=query,
        columns=["content"],
        num_results=TOP_K_BRD
    )
    return "\n\n".join(r["content"] for r in res["result"]["data_array"])


In [0]:
# llm setup
llm = ChatDatabricks(
    endpoint=LLM_ENDPOINT,
    temperature=0.3
)


In [0]:
# brd generation prompt

BRD_PROMPT = PromptTemplate(
    input_variables=["system_prompt", "notebook_code", "notebook_ctx", "brd_ctx"],
    template="""
{system_prompt}

==============================
NOTEBOOK CONTENT
==============================
{notebook_code}

==============================
NOTEBOOK LOGIC CONTEXT (VECTOR SEARCH)
==============================
{notebook_ctx}

==============================
GOLD BRD STRUCTURE CONTEXT (VECTOR SEARCH)
==============================
{brd_ctx}

==============================
TASK
==============================
Generate the Business Requirements Document (BRD)
strictly following the system instructions.
"""
)


In [0]:
# docx generator

def save_docx_from_html(html: str, file_name: str):
    from docx import Document
    from bs4 import BeautifulSoup

    soup = BeautifulSoup(html, "html.parser")
    doc = Document()

    for elem in soup.find_all(["h1","h2","h3","h4","p"]):
        doc.add_paragraph(elem.get_text())

    path = f"{OUTPUT_VOLUME}/{file_name}"
    doc.save(path)
    return path


In [0]:
NOTEBOOK_PATH = dbutils.widgets.get("notebook_path").strip()
BRD_FILE_NAME = dbutils.widgets.get("brd_file_name").strip()

if not NOTEBOOK_PATH:
    raise ValueError("Please provide notebook_path")

if not BRD_FILE_NAME:
    raise ValueError("Please provide brd_file_name")

print("Notebook path:", NOTEBOOK_PATH)
print("BRD file name:", BRD_FILE_NAME)



Notebook path: /Workspace/Users/shivam.kumar@dvn.com/Company
BRD file name: BRD_BI_REPORTING_COMPANY_TEST


In [0]:
# main execution cell   
# =========================
# INPUT
# =========================
NOTEBOOK_PATH = dbutils.widgets.get("notebook_path")


In [0]:
def retrieve_notebook_context(query: str):
    index = vsc.get_index(
        endpoint_name=VS_ENDPOINT_NAME,
        index_name=NOTEBOOK_INDEX
    )
    res = index.similarity_search(
        query_text=query,
        columns=["content"],
        num_results=TOP_K_NOTEBOOK
    )
    # Each element in data_array is a list; content is the first element
    return "\n\n".join(r[0] for r in res["result"]["data_array"])

In [0]:
# similarity search using Vector search index for notebook above and brd

def retrieve_brd_context(query: str):
    index = vsc.get_index(
        endpoint_name=VS_ENDPOINT_NAME,
        index_name=BRD_INDEX
    )
    res = index.similarity_search(
        query_text=query,
        columns=["content"],
        num_results=TOP_K_BRD
    )
    # Each element in data_array is a list; content is the first element
    return "\n\n".join(r[0] for r in res["result"]["data_array"])

In [0]:
print("Reading notebook...")
notebook_code = read_notebook_content(NOTEBOOK_PATH)

print("Retrieving vector contexts...")
notebook_ctx = retrieve_notebook_context(notebook_code[:2000])
brd_ctx = retrieve_brd_context("BRD structure and mapping rules")

print("Calling LLM...")
response = llm.invoke(
    BRD_PROMPT.format(
        system_prompt=SYSTEM_PROMPT,
        notebook_code=notebook_code,
        notebook_ctx=notebook_ctx,
        brd_ctx=brd_ctx
    )
)

brd_markdown = response.content


Reading notebook...
Retrieving vector contexts...
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Calling LLM...


In [0]:
# preview brd
displayHTML(f"<pre>{brd_markdown}</pre>")


Document Title,BRD – Enterprise Company Master Data (ENTERPRISE.COMPANY) and Reporting Company (REPORTING.COMPANY)
Domain,Enterprise
Description,Company master data information – A Company is an organizational unit that represents a business organization for which financial statements can be drawn.
Precondition,FINANCE_AND_ACCOUNTING.COMPANY_HIERARCHY
Postcondition,N/A
Source Notebook Dependencies,/Apps/BIReporting_S4/Enterprise/Geography_Country
Generated On,TBD
Author,TBD
Audience,Purpose
Business Stakeholders,Understand business rules and outputs for Company master and reporting Company datasets.
Data Engineering / ETL Developers,Implement and maintain the pipeline logic as defined in the notebook.


In [0]:

import docx

In [0]:
# SAVE DOCX
OUTPUT_VOLUME = "/Volumes/tmp/tmp/brd_agent/output_brd"

# Ensure directory exists
import os
os.makedirs(OUTPUT_VOLUME, exist_ok=True)

print("Output directory ready:", OUTPUT_VOLUME)



Output directory ready: /Volumes/tmp/tmp/brd_agent/output_brd


In [0]:
def save_docx_from_html(html: str, base_file_name: str):
    from docx import Document
    from bs4 import BeautifulSoup
    from io import BytesIO
    import base64

    # Build DOCX in memory
    soup = BeautifulSoup(html, "html.parser")
    doc = Document()

    for elem in soup.find_all(["h1", "h2", "h3", "h4", "p"]):
        doc.add_paragraph(elem.get_text())

    buffer = BytesIO()
    doc.save(buffer)
    buffer.seek(0)

    # Encode to base64 for dbutils.fs.put
    b64_content = base64.b64encode(buffer.read()).decode("utf-8")

    # Final path in Volume
    final_path = f"/Volumes/tmp/tmp/brd_agent/output_brd/{base_file_name}.docx"

    # Write directly (NO FILE SELECT PERMISSION NEEDED)
    dbutils.fs.put(
        final_path,
        b64_content,
        overwrite=True
    )

    return final_path


In [0]:
def save_brd_as_html(html_content: str, base_file_name: str):
    final_path = f"/Volumes/tmp/tmp/brd_agent/output_brd/{base_file_name}.html"

    dbutils.fs.put(
        final_path,
        html_content,
        overwrite=True
    )

    return final_path


In [0]:
html_path = save_brd_as_html(
    html_content=brd_markdown,   # your BRD output
    base_file_name=BRD_FILE_NAME
)

print("BRD HTML generated successfully:")
print(html_path)


Wrote 24631 bytes.
BRD HTML generated successfully:
/Volumes/tmp/tmp/brd_agent/output_brd/BRD_BI_REPORTING_COMPANY_TEST.html


In [0]:
#final save

docx_path = save_docx_from_html(
    html=brd_markdown,
    base_file_name=BRD_FILE_NAME
)

print("DOCX generated successfully:")
print(docx_path)


Wrote 50040 bytes.
DOCX generated successfully:
/Volumes/tmp/tmp/brd_agent/output_brd/BRD_BI_REPORTING_COMPANY_TEST.docx
